Since we only implement K+1 to 2K+1 order compensation (not $log(1/\epsilon)$),
we cannot achieve log precision.
Scaling to verify: 1. commutator scaling, 2. time dependence

Not sure if applies to J \neq h

In [1]:
import numpy as np
from scipy.linalg import expm

# Define Pauli matrices
X = np.array([[0, 1], [1, 0]])  # Pauli-X matrix
Y = np.array([[0, -1j], [1j, 0]])  # Pauli-Y matrix
Z = np.array([[1, 0], [0, -1]])  # Pauli-Z matrix
I = np.eye(2)  # Identity matrix


def commutator(A, B):
    """Compute the commutator [A, B] = AB - BA"""
    return A @ B - B @ A

# Parameters
N = 6  # Number of spins
J = 1  # Interaction strength
h = 1  # Transverse field strength
T = 1  # Evolution time

g = 2 * (J + h)  # extensive parameter
k = 2  # 2-local Hamiltonian

r = 10  # Trotter steps
t = T / r  # step size

K = 1

# target accuracy
epsilon = 0.01

print('time step:', t)

time step: 0.1


In [2]:
Heisenberg = False
if Heisenberg:
    p_Pauli = np.array([J, J, J, h]) / (3 * J + h)
else:
    p_Pauli = np.array([J, 0, 0, h]) / (J + h)


def XX_term(index):
    if index < N - 1:
        return J * np.kron(
            np.eye(2**index), np.kron(np.kron(X, X), np.eye(2 ** (N - index - 2)))
        )
    elif index == N - 1:  # periodic boundary condition
        return np.kron(X, np.kron(np.eye(2 ** (N - 2)), X))
    else:
        raise IndexError("out of range")


def YY_term(index):
    if not Heisenberg:
        return np.zeros((2**N, 2**N), dtype=complex)
    if index < N - 1:
        return J * np.kron(
            np.eye(2**index), np.kron(np.kron(Y, Y), np.eye(2 ** (N - index - 2)))
        )
    elif index == N - 1:  # periodic boundary condition
        return np.kron(Y, np.kron(np.eye(2 ** (N - 2)), Y))
    else:
        raise IndexError("out of range")


def ZZ_term(index):
    if not Heisenberg:
        return np.zeros((2**N, 2**N), dtype=complex)
    if index < N - 1:
        return J * np.kron(
            np.eye(2**index), np.kron(np.kron(Z, Z), np.eye(2 ** (N - index - 2)))
        )
    elif index == N - 1:  # periodic boundary condition
        return np.kron(Z, np.kron(np.eye(2 ** (N - 2)), Z))
    else:
        raise IndexError("out of range")


def Z_term(index):
    if index <= N - 1:
        return h * np.kron(np.eye(2**index), np.kron(Z, np.eye(2 ** (N - index - 1))))
    else:
        raise IndexError("out of range")


def padding_term(index):
    if index >= N - 1:
        return np.eye(2**N)
    else:
        raise IndexError("illegal padding!")

In [8]:
# construct product formula
A = np.zeros((2**N, 2**N), dtype=complex)
B = np.zeros((2**N, 2**N), dtype=complex)

for index in range(0, N, 2):
    A += XX_term(index) + YY_term(index) + ZZ_term(index) + Z_term(index)

for index in range(1, N, 2):
    B += XX_term(index) + YY_term(index) + ZZ_term(index) + Z_term(index)
# note the order of exp(A) and exp(B)
V_exact = expm(-1j * (A + B) * t) @ expm(1j * A * t) @ expm(1j * B * t)
S = expm(-1j * B * t) @ expm(-1j * A * t)
# apply A first, V_exact @ S = I


def pauli_l1_norm(matrix, num_qubits):
    """Compute l1 norm of matrix coefficients in n-qubit Pauli basis."""
    single_basis = [I, X, Y, Z]
    basis = [np.array([[1]], dtype=complex)]
    for _ in range(num_qubits):
        next_basis = []
        for left in basis:
            for right in single_basis:
                next_basis.append(np.kron(left, right))
        basis = next_basis
    scale = 2**num_qubits
    l1_norm = 0.0
    for P in basis:
        coeff = np.trace(P.conj().T @ matrix) / scale
        l1_norm += abs(coeff)
    return float(l1_norm)


C1 = commutator(B, A)
C2 = 1j * (2 * commutator(B, commutator(A, B)) + commutator(A, commutator(A, B)))
c1_l1 = pauli_l1_norm(C1, N)
c2_l1 = pauli_l1_norm(C2, N)

eta2 = c1_l1 * (t**2) / 2
eta3 = c2_l1 * (t**3) / 6
eta_sum = eta2 + eta3
p_s = [eta2 / eta_sum, eta3 / eta_sum]
theta = np.arctan(eta_sum)

print("C1 l1:", c1_l1)
print("C2 l1:", c2_l1)
print("eta2, eta3:", eta2, eta3)
print("p_s:", p_s)
print("theta:", theta)





def compensation_unitary(W, theta, atol=1e-10):
    """Build compensation unitary from sampled W.

    If W is Hermitian, use exp(i theta W).
    If W is anti-Hermitian, use exp(theta W).
    """
    hermitian_err = np.linalg.norm(W - W.conj().T, ord="fro")
    antihermitian_err = np.linalg.norm(W + W.conj().T, ord="fro")
    if hermitian_err <= atol:
        return expm(1j * theta * W)
    if antihermitian_err <= atol:
        return expm(theta * W)
    raise ValueError(
        f"sampled W is neither Hermitian nor anti-Hermitian (herm_err={hermitian_err:.3e}, antiherm_err={antihermitian_err:.3e})"
    )

# K = 1, sample from s = 2, 3
def NCC_sampling(trials = 100):
    rng = np.random.default_rng(seed=7)

    # NCC sampling
    s_list = [2, 3]

    V_list = []
    for _ in range(trials):
        s = int(rng.choice(s_list, p=p_s))

        j = rng.choice(np.arange(1, N, 2))
        W = rng.choice([XX_term(j), YY_term(j), ZZ_term(j), Z_term(j)], p=p_Pauli)
        j1 = rng.choice([j - 1, j + 1]) % N
        P1 = rng.choice(
            [XX_term(j1), YY_term(j1), ZZ_term(j1), Z_term(j1)], p=p_Pauli
        )
        b1 = rng.choice([0, 1])
        if b1 == 0:
            W = P1 @ W
        else:
            W = -W @P1

        if s == 3:
            if rng.random() <= 1 / 3:
                j2 = rng.choice([j - 2, j, j + 2]) % N
            else:
                j2 = rng.choice([j - 3, j - 1, j + 1]) % N
            P2 = rng.choice(
                [XX_term(j2), YY_term(j2), ZZ_term(j2), Z_term(j2)], p=p_Pauli
            )
            b2 = rng.choice([0, 1])

            if b2 == 0:
                W = P2 @ W
            else:
                W = -W @ P2

        V_list.append(compensation_unitary(W, theta))
        
    V_average = sum(V_list) / trials
    
    return V_list, V_average

In [9]:
print("single Trotter step error before compensation:\n", np.linalg.norm(S - expm(-1j * (A + B) * t), 2))

V_list, V_average = NCC_sampling(trials=1000)
print(
    "single Trotter step error after compensation:\n",
    np.linalg.norm(V_average - V_exact, 2),
)

single Trotter step error before compensation:
 0.039858721289656174
single Trotter step error after compensation:
 0.2824415054581952


In [ ]:
# construct product formula
A = np.zeros((2**N, 2**N), dtype=complex)
B = np.zeros((2**N, 2**N), dtype=complex)

for index in range(0, N, 2):
    A += XX_term(index) + YY_term(index) + ZZ_term(index) + Z_term(index)

for index in range(1, N, 2):
    B += XX_term(index) + YY_term(index) + ZZ_term(index) + Z_term(index)
# note the order of exp(A) and exp(B)
S = expm(-1j * B * t) @ expm(-1j * A * t)
V_exact = expm(-1j * (A + B) * t) @ expm(1j * A * t) @ expm(1j * B * t)
S_r = np.eye(2**N, dtype=complex)
for _ in range(r):
    S_r = S @ S_r
evolution_exact = expm(-1j * (A + B) * t * r)

def multi_step_NCC_sampling(trials):
    rng = np.random.default_rng(seed=7)

    # NCC sampling
    s_list = [2, 3]

    evolution_list = []
    for _ in range(trials):
        evolution = np.eye(2**N, dtype=complex)
        for _ in range(r):
            s = int(rng.choice(s_list, p=p_s))

            j = rng.choice(np.arange(1, N, 2))
            W = rng.choice([XX_term(j), YY_term(j), ZZ_term(j), Z_term(j)], p=p_Pauli)
            j1 = rng.choice([j - 1, j + 1]) % N
            P1 = rng.choice([XX_term(j1), YY_term(j1), ZZ_term(j1), Z_term(j1)], p=p_Pauli)
            b1 = rng.choice([0, 1])
            if b1 == 0:
                W = P1 @ W
            else:
                W = -W @ P1

            if s == 3:
                if rng.random() <= 1 / 3:
                    j2 = rng.choice([j - 2, j, j + 2]) % N
                else:
                    j2 = rng.choice([j - 3, j - 1, j + 1]) % N
                P2 = rng.choice(
                    [XX_term(j2), YY_term(j2), ZZ_term(j2), Z_term(j2)], p=p_Pauli
                )
                b2 = rng.choice([0, 1])

                if b2 == 0:
                    W = P2 @ W
                else:
                    W = -W @ P2

            evolution = compensation_unitary(W, theta) @ S @ evolution
            
        evolution_list.append(evolution)
        
    evolution_average = sum(evolution_list) / trials

    return evolution_list, evolution_average

In [ ]:
print(
    "total evolution error before compensation:\n",
    np.linalg.norm(S_r - evolution_exact, 2),
)

evolution_list, evolution_average = multi_step_NCC_sampling(trials=100)
print(
    "total evolution error after compensation:\n",
    np.linalg.norm(evolution_average - evolution_exact, 2),
)

total evolution error before compensation:
 0.13669481049118504
total evolution error after compensation:
 1.1774927141545621
